In [1]:
# Optional: Create the CSV file using Python
content = """
transaction_id,date,customer_name,product,quantity,unit_price,region
T001,2024-06-01,alice smith,Widget A,3,25.00,east
T002,2024-06-01,Bob Jones,Widget B,1,50.00,West
T003,2024-06-02,alice smith,Widget A,2,25.00,EAST
T004,2024-06-02,Charlie Lee,,5,10.00,south
T005,2024-06-03,Bob Jones,Widget C,-1,30.00,West
T006,2024-06-03,alice smith,Widget A,3,25.00,east
T007,2024-06-04,,Widget B,2,50.00,North
T008,2024-06-04,Diana Ross,Widget A,0,25.00,east
T009,2024-06-05,Bob Jones,Widget B,1,50.00,West
T010,2024-06-05,Bob Jones,Widget B,1,50.00,West
"""

with open("raw_transactions.csv", "w") as file:
    file.write(content)

print("raw_transactions.csv created successfully!")

raw_transactions.csv created successfully!


In [8]:
import pandas as pd

def extract(file):
    """
    EXTRACT PHASE: Read raw data from source file.
    This is the Bronze layer — data as-is from the source.
    
    Args:
        file_path (str): Path to the raw CSV file
    
    Returns:
        pd.DataFrame: Raw data as a DataFrame
    """
    print("=" * 50)
    print("EXTRACT PHASE (Bronze Layer)")
    print("=" * 50)
    
    # Read the raw CSV
    df = pd.read_csv(file)
    
    # Print extraction summary
    print(f"Source file: {file}")
    print(f"Rows extracted: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(f"\nColumn names: {list(df.columns)}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    
    return df


In [10]:
raw_df = extract("raw_transactions.csv")


EXTRACT PHASE (Bronze Layer)
Source file: raw_transactions.csv
Rows extracted: 10
Columns: 7

Column names: ['transaction_id', 'date', 'customer_name', 'product', 'quantity', 'unit_price', 'region']

Data types:
transaction_id     object
date               object
customer_name      object
product            object
quantity            int64
unit_price        float64
region             object
dtype: object

Missing values:
transaction_id    0
date              0
customer_name     1
product           1
quantity          0
unit_price        0
region            0
dtype: int64

First 3 rows:
  transaction_id        date customer_name   product  quantity  unit_price  \
0           T001  2024-06-01   alice smith  Widget A         3        25.0   
1           T002  2024-06-01     Bob Jones  Widget B         1        50.0   
2           T003  2024-06-02   alice smith  Widget A         2        25.0   

  region  
0   east  
1   West  
2   EAST  


In [11]:
def transform(df):
    """
    TRANSFORM PHASE: Clean, validate, and standardize data.
    This is the Silver layer — data is trustworthy and consistent.
    
    Args:
        df (pd.DataFrame): Raw data from Extract phase
    
    Returns:
        pd.DataFrame: Cleaned and validated data
    """
    print("\n" + "=" * 50)
    print("TRANSFORM PHASE (Silver Layer)")
    print("=" * 50)
    
    rows_before = len(df)
    
    # --- Step 3a: Drop rows with missing critical fields ---
    print("\n--- Handling Missing Values ---")
    critical_columns = ["customer_name", "product"]
    missing_before = df[critical_columns].isnull().sum()
    print(f"Missing values in critical columns:\n{missing_before}")
    
    df = df.dropna(subset=critical_columns)
    print(f"Rows dropped due to nulls: {rows_before - len(df)}")
    
    # --- Step 3b: Remove exact duplicate rows ---
    print("\n--- Removing Duplicates ---")
    duplicates_count = df.duplicated().sum()
    print(f"Duplicate rows found: {duplicates_count}")
    
    df = df.drop_duplicates()
    print(f"Rows after removing duplicates: {len(df)}")
    
    # --- Step 3c: Validate quantity (must be > 0) ---
    print("\n--- Validating Quantity ---")
    invalid_qty = df[df["quantity"] <= 0]
    print(f"Rows with invalid quantity (<= 0): {len(invalid_qty)}")
    if len(invalid_qty) > 0:
        print(f"  Invalid rows:\n{invalid_qty[['transaction_id', 'quantity']]}")
    
    df = df[df["quantity"] > 0]
    print(f"Rows after quantity validation: {len(df)}")
    
    # --- Step 3d: Standardize text fields ---
    print("\n--- Standardizing Text Fields ---")
    df["customer_name"] = df["customer_name"].str.strip().str.title()
    df["region"] = df["region"].str.strip().str.title()
    df["product"] = df["product"].str.strip()
    
    print(f"Unique regions after standardization: {sorted(df['region'].unique())}")
    print(f"Unique customers after standardization: {sorted(df['customer_name'].unique())}")
    
    # --- Step 3e: Add computed column ---
    print("\n--- Adding Computed Columns ---")
    df["total_amount"] = df["quantity"] * df["unit_price"]
    print(f"Added 'total_amount' column (quantity * unit_price)")
    
    # --- Summary ---
    rows_after = len(df)
    print(f"\n--- Transform Summary ---")
    print(f"Rows before: {rows_before}")
    print(f"Rows after: {rows_after}")
    print(f"Rows removed: {rows_before - rows_after}")
    print(f"Columns: {list(df.columns)}")
    
    return df


In [12]:
# test the transform function
clean_df = transform(raw_df)
print("\nCleaned data:")
print(clean_df)



TRANSFORM PHASE (Silver Layer)

--- Handling Missing Values ---
Missing values in critical columns:
customer_name    1
product          1
dtype: int64
Rows dropped due to nulls: 2

--- Removing Duplicates ---
Duplicate rows found: 0
Rows after removing duplicates: 8

--- Validating Quantity ---
Rows with invalid quantity (<= 0): 2
  Invalid rows:
  transaction_id  quantity
4           T005        -1
7           T008         0
Rows after quantity validation: 6

--- Standardizing Text Fields ---
Unique regions after standardization: ['East', 'West']
Unique customers after standardization: ['Alice Smith', 'Bob Jones']

--- Adding Computed Columns ---
Added 'total_amount' column (quantity * unit_price)

--- Transform Summary ---
Rows before: 10
Rows after: 6
Rows removed: 4
Columns: ['transaction_id', 'date', 'customer_name', 'product', 'quantity', 'unit_price', 'region', 'total_amount']

Cleaned data:
  transaction_id        date customer_name   product  quantity  unit_price  \
0        

In [13]:
# create the load function
def load(df, silver_path, gold_path):
    """
    LOAD PHASE: Save cleaned data and produce analytics-ready output.
    Silver layer: cleaned, validated data.
    Gold layer: aggregated, business-ready data.
    
    Args:
        df (pd.DataFrame): Cleaned data from Transform phase
        silver_path (str): Output path for Silver data (cleaned)
        gold_path (str): Output path for Gold data (aggregated)
    """
    print("\n" + "=" * 50)
    print("LOAD PHASE")
    print("=" * 50)
    
    # --- Save Silver (cleaned data) ---
    print("\n--- Silver Layer (Cleaned Data) ---")
    df.to_csv(silver_path, index=False)
    print(f"Saved {len(df)} rows to: {silver_path}")
    
    # --- Build Gold (aggregated analytics) ---
    print("\n--- Gold Layer (Aggregated Analytics) ---")
    
    gold_df = df.groupby(["region", "product"]).agg(
        total_quantity=("quantity", "sum"),
        total_revenue=("total_amount", "sum"),
        transaction_count=("transaction_id", "count"),
        avg_unit_price=("unit_price", "mean")
    ).reset_index()
    
    # Round numeric columns
    gold_df["total_revenue"] = gold_df["total_revenue"].round(2)
    gold_df["avg_unit_price"] = gold_df["avg_unit_price"].round(2)
    
    # Sort by total revenue descending
    gold_df = gold_df.sort_values("total_revenue", ascending=False)
    
    gold_df.to_csv(gold_path, index=False)
    print(f"Saved {len(gold_df)} aggregated rows to: {gold_path}")
    
    # --- Print Gold summary ---
    print(f"\nGold layer output:")
    print(gold_df.to_string(index=False))
    
    # --- Pipeline summary ---
    print(f"\n--- Load Summary ---")
    print(f"Silver output: {silver_path} ({len(df)} rows)")
    print(f"Gold output: {gold_path} ({len(gold_df)} rows)")
    print(f"Total revenue: {gold_df['total_revenue'].sum():,.2f}")


In [14]:
# test the load function
load(clean_df, "silver_transactions.csv", "gold_transactions.csv")



LOAD PHASE

--- Silver Layer (Cleaned Data) ---
Saved 6 rows to: silver_transactions.csv

--- Gold Layer (Aggregated Analytics) ---
Saved 2 aggregated rows to: gold_transactions.csv

Gold layer output:
region  product  total_quantity  total_revenue  transaction_count  avg_unit_price
  East Widget A               8          200.0                  3            25.0
  West Widget B               3          150.0                  3            50.0

--- Load Summary ---
Silver output: silver_transactions.csv (6 rows)
Gold output: gold_transactions.csv (2 rows)
Total revenue: 350.00


In [15]:
# create pipeline function
def run_etl_pipeline(input_file, silver_output, gold_output):
    """
    Run the complete ETL pipeline: Extract → Transform → Load.
    
    Args:
        input_file (str): Path to raw input CSV (Bronze)
        silver_output (str): Path for cleaned output (Silver)
        gold_output (str): Path for aggregated output (Gold)
    """
    print("*" * 60)
    print("  ETL PIPELINE START")
    print("*" * 60)
    
    # Phase 1: Extract (Bronze)
    raw_df = extract(input_file)
    
    # Phase 2: Transform (Silver)
    clean_df = transform(raw_df)
    
    # Phase 3: Load (Silver + Gold)
    load(clean_df, silver_output, gold_output)
    
    print("\n" + "*" * 60)
    print("  ETL PIPELINE COMPLETE")
    print("*" * 60)


In [16]:
# execute the pipeline
run_etl_pipeline(
    input_file="raw_transactions.csv",
    silver_output="silver_transactions.csv",
    gold_output="gold_transactions.csv"
)


************************************************************
  ETL PIPELINE START
************************************************************
EXTRACT PHASE (Bronze Layer)
Source file: raw_transactions.csv
Rows extracted: 10
Columns: 7

Column names: ['transaction_id', 'date', 'customer_name', 'product', 'quantity', 'unit_price', 'region']

Data types:
transaction_id     object
date               object
customer_name      object
product            object
quantity            int64
unit_price        float64
region             object
dtype: object

Missing values:
transaction_id    0
date              0
customer_name     1
product           1
quantity          0
unit_price        0
region            0
dtype: int64

First 3 rows:
  transaction_id        date customer_name   product  quantity  unit_price  \
0           T001  2024-06-01   alice smith  Widget A         3        25.0   
1           T002  2024-06-01     Bob Jones  Widget B         1        50.0   
2           T003  2024-06-02  

In [17]:
# verify outputs:
# Verify Silver
print("=== SILVER OUTPUT ===")
silver = pd.read_csv("silver_transactions.csv")
print(silver)
print(f"Shape: {silver.shape}")

# Verify Gold
print("\n=== GOLD OUTPUT ===")
gold = pd.read_csv("gold_transactions.csv")
print(gold)
print(f"Shape: {gold.shape}")


=== SILVER OUTPUT ===
  transaction_id        date customer_name   product  quantity  unit_price  \
0           T001  2024-06-01   Alice Smith  Widget A         3        25.0   
1           T002  2024-06-01     Bob Jones  Widget B         1        50.0   
2           T003  2024-06-02   Alice Smith  Widget A         2        25.0   
3           T006  2024-06-03   Alice Smith  Widget A         3        25.0   
4           T009  2024-06-05     Bob Jones  Widget B         1        50.0   
5           T010  2024-06-05     Bob Jones  Widget B         1        50.0   

  region  total_amount  
0   East          75.0  
1   West          50.0  
2   East          50.0  
3   East          75.0  
4   West          50.0  
5   West          50.0  
Shape: (6, 8)

=== GOLD OUTPUT ===
  region   product  total_quantity  total_revenue  transaction_count  \
0   East  Widget A               8          200.0                  3   
1   West  Widget B               3          150.0                  3   

   a

In [ ]:
# Step 3: Build the Decision Table (10 minutes)
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| Scenario | Class- | Transform  | Key Driver | Pros                | Cons                 | Justification           | Alternative Considered   | Pattern                     |
|          | ific- | Location   |            |                     |                      |                         |                          |                             |
|          | ation  |            |            |                     |                      |                         |                          |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1. PII   | ETL    | External   | Compliance | Zero raw PII         | Can't reprocess      | PII cannot sit raw       | ELT with dynamic         | Compliance-driven ETL       |
| CRM      |        | (Python/   | (no raw    | ever enters         | raw data if hashing  | in warehouse even        | masking - rejected        | (#1)                        |
|          |        | Spark)     | PII in     | Snowflake; fully    | algorithm needs to   | briefly                  | due to raw PII exposure   |                             |
|          |        |            | warehouse) | compliant with       | change               |                         | in staging                |                             |
|          |        |            |            | GDPR/CCPA            |                      |                         |                          |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 2. Click- | ELT    | In-        | Volume +   | Raw data preserved   | Storage costs        | BigQuery massively       | ETL with Spark            | Cloud-warehouse ELT         |
| stream    |        | warehouse   | raw data   | for future analyses; | accumulate; query    | parallel; analysts       | streaming - rejected      | (#2)                        |
|           |        | (BigQuery   | exploration| BigQuery handles     | costs high if        | need raw access          | due to complexity         |                             |
|           |        | SQL)        |            | scale effortlessly   | scanning full table  |                         | without benefit           |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 3. Fin-   | ETL    | External    | Complex    | Complex matching     | Extra hop adds       | PostgreSQL limited       | ELT with stored           | Complex logic forces        |
| ancial    |        | (Python)    | logic      | implemented in       | latency; Python      | compute; fuzzy matching  | procedures - rejected     | external processing         |
|           |        |            | (fuzzy     | Python libraries     | env maintenance      | impossible in SQL        | due to SQL limitations    | (#4)                        |
|           |        |            | matching)  |                      |                      |                         |                          |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 4. IoT    | Hybrid | Both       | Dual       | Best of both         | Pipeline complexity  | 50M events/day too high  | Single target with        | Hybrid triggers             |
|           |        | (Stream +   | destina-   | worlds - raw for     | doubled; streaming   | for direct Snowflake     | Snowflake views -         | (#3)                        |
|           |        | Warehouse)  | tions +    | ML, aggregates       | infra maintenance    | queries; ML needs raw,   | rejected due to speed     |                             |
|           |        |            | volume     | for ops speed        |                      | ops needs speed          | and cost concerns         |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 5. HR     | ETL    | External    | Extreme    | Individual salaries   | HR leadership can't  | Salary data highly       | ELT with Redshift         | Compliance-driven ETL       |
|           |        | (Python/    | sensitivity| NEVER enter          | do ad-hoc on         | sensitive; safer to      | column security -         | (#1)                        |
|           |        | Spark)      |            | warehouse - zero     | individual data      | never ingest than        | rejected due to           |                             |
|           |        |            |            | internal exposure    | (but shouldn't)      | rely on permissions      | exposure risk             |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 6. Product | ELT    | In-        | Simplicity | Simple pipeline,     | Loads slightly       | Simple CDC cleaning;     | ETL with Python -         | Cloud-warehouse ELT         |
| Catalog    |        | warehouse   | + Snowflake| leverages Snowflake's| more data (but       | Snowflake has plenty     | rejected as                | (#2)                        |
|            |        | (Snowflake  | power      | compute, easy to     | at 10K records,      | of compute for simple    | overengineering           |                             |
|            |        | SQL)        |            | modify transformations| irrelevant)         | tasks                    |                          |                             |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Step 4: Design a Hybrid Architecture (10 minutes)
## HYBRID ARCHITECTURE DESIGN - COMPLETE OVERVIEW

### Architecture Component Matrix

--------------------------------------------------------------------------------------------------------------------
| Layer           | Component          | Technology          | Purpose                          | Scenarios Used |
--------------------------------------------------------------------------------------------------------------------
| **Ingestion**   | API Connector      | Azure Data Factory  | Scheduled pulls from REST APIs   | 1, 5           |
|                 | Stream Processor   | Kafka/Spark Streaming| Real-time event consumption      | 2, 4           |
|                 | CDC Capture        | Debezium            | Change data capture from DB       | 6              |
|                 | File Ingestion     | ADF / Storage SDK   | Batch file processing             | 3              |
|                 | MQTT Subscriber    | MQTT Client + Spark | IoT telemetry ingestion           | 4              |
--------------------------------------------------------------------------------------------------------------------
| **External     | PII Masking Engine  | Python/PySpark      | Hash/encrypt PII before loading   | 1              |
| Transform**    | Fuzzy Matcher       | Python + fuzzywuzzy | Complex reconciliation logic       | 3              |
| (Pre-load)     | Salary Aggregator   | Python/PySpark      | Aggregate sensitive HR data       | 5              |
|                | Stream Aggregator   | Spark Streaming     | Real-time IoT aggregations        | 4              |
--------------------------------------------------------------------------------------------------------------------
| **Storage      | Bronze Zone (Raw)   | S3/ADLS/GCS         | Raw immutable data                | 2, 3, 4        |
| (Data Lake)**  | Silver Zone (Clean) | S3/ADLS/GCS         | Cleaned, validated data           | 1, 6           |
|                | Gold Zone (Agg)     | S3/ADLS/GCS         | Aggregated, business-ready        | 4, 5           |
--------------------------------------------------------------------------------------------------------------------
| **Storage      | Raw Tables          | Snowflake/BigQuery  | Raw data for exploration          | 2, 6           |
| (Warehouse)**  | Staging Tables      | Snowflake/BigQuery  | Landing area for external transforms| 1, 3, 5       |
|                | Curated Tables      | Snowflake/BigQuery  | Cleaned, modeled data              | 1, 2, 4, 6     |
|                | Aggregated Tables   | Snowflake/Redshift  | Pre-joined, pre-aggregated         | 4, 5           |
--------------------------------------------------------------------------------------------------------------------
| **Governance** | Metadata Catalog    | Unity Catalog       | Central metadata management        | All            |
|                | Data Lineage        | Purview/Colibra     | Track data origins                 | All            |
|                | Access Control      | RBAC + ABAC         | Fine-grained permissions           | 1, 5           |
--------------------------------------------------------------------------------------------------------------------
| **Serving**    | BI Dashboards       | Power BI/Looker     | Visualizations and reporting       | 1, 2, 4, 5, 6  |
|                | ML Platform         | Databricks/MLflow   | Model training & inference         | 2, 4           |
|                | Ad-hoc Analytics    | SQL Interfaces      | Direct query access                | 2, 3, 6        |
--------------------------------------------------------------------------------------------------------------------

### Data Flow by Scenario

--------------------------------------------------------------------------------------------------------------------
| Scenario | Source      | Ingestion Path | Transform Path 1 | Transform Path 2 | Storage Path    | Serving Path   |
--------------------------------------------------------------------------------------------------------------------
| 1 - PII  | Salesforce  | ADF Scheduled  | Python PII       | N/A              | Silver Lake →   | Power BI       |
| CRM      | API         | → ADF Staging  | Masking (Ext)    |                  | Snowflake WH    | Dashboards     |
--------------------------------------------------------------------------------------------------------------------
| 2 - Click-| Kafka       | Spark Streaming| Load Raw to      | BigQuery SQL     | BigQuery Raw →  | Product        |
| stream   | Topic       | → BigQuery Raw | BigQuery         | Transform (WH)   | Curated Tables  | Analysts + ML  |
--------------------------------------------------------------------------------------------------------------------
| 3 - Fin- | Banking CSV | ADF Batch      | Python Fuzzy     | N/A              | Bronze Lake →   | PostgreSQL     |
| ancial   | Files       | → Bronze Lake  | Matching (Ext)   |                  | PostgreSQL      | Reports        |
--------------------------------------------------------------------------------------------------------------------
| 4 - IoT  | MQTT Broker | Spark Streaming| Path A: Raw to   | Path B: Spark    | Bronze Lake +   | Ops Dashboards |
|          |             | → Split Path   | Bronze Lake      | Stream Agg (Ext) | Snowflake Agg   | + ML Models    |
--------------------------------------------------------------------------------------------------------------------
| 5 - HR   | HR API      | ADF Scheduled  | Python Salary    | N/A              | Agg Only →      | HR Leadership  |
|          |             | → Python Env   | Agg (Ext)        |                  | Redshift        | Dashboards     |
--------------------------------------------------------------------------------------------------------------------
| 6 - Prod | PostgreSQL  | Debezium CDC   | Load Raw to      | Snowflake SQL    | Snowflake Raw → | BI Team        |
| Catalog  |             | → Kafka → WH   | Snowflake        | Clean (WH)       | Curated Tables  | SQL Queries    |
--------------------------------------------------------------------------------------------------------------------

### Pipeline Configuration Matrix

--------------------------------------------------------------------------------------------------------------------
| Scenario | Pipeline Type | Frequency  | Compute Used      | Data Volume | SLA        | Critical Priority  |
--------------------------------------------------------------------------------------------------------------------
| 1 - PII  | Batch ETL     | Daily      | ADF + Databricks  | 50K rec     | 4 hours    | Medium - Compliance|
--------------------------------------------------------------------------------------------------------------------
| 2 - Click-| Streaming ELT | 24/7       | Kafka + BigQuery  | 10M/day     | < 1 min    | High - Real-time   |
| stream   |               |            |                   |             |            | Analytics          |
--------------------------------------------------------------------------------------------------------------------
| 3 - Fin- | Batch ETL     | Daily      | ADF + Python      | 100K rec    | 6 hours    | High - Financial   |
| ancial   |               |            |                   |             |            | Accuracy           |
--------------------------------------------------------------------------------------------------------------------
| 4 - IoT  | Streaming     | 24/7       | Spark + S3 +      | 50M/day     | < 30 sec   | High - Ops Monitoring
|          | Hybrid        |            | Snowflake         |             |            |                    |
--------------------------------------------------------------------------------------------------------------------
| 5 - HR   | Batch ETL     | Monthly    | ADF + Python      | 5K rec      | 24 hours   | Critical -         |
|          |               |            |                   |             |            | Compliance         |
--------------------------------------------------------------------------------------------------------------------
| 6 - Prod | CDC ELT       | Continuous | Debezium +        | 10K changes  | < 5 min    | Medium - BI        |
| Catalog  |               |            | Snowflake         | /day         |            | Consistency        |
--------------------------------------------------------------------------------------------------------------------

### Storage Layer Detailed Design

--------------------------------------------------------------------------------------------------------------------
| Storage Tier | Zone        | Format      | Retention | Scenarios      | Access Pattern      | Consumers        |
--------------------------------------------------------------------------------------------------------------------
| **Data Lake**| Bronze      | JSON/CSV/   | 90 days   | 2, 3, 4        | Write once,         | ML Engineers,    |
| (S3/ADLS/GCS) | (Raw)       | Parquet     |           |                | read rarely         | Auditors         |
|              | Silver      | Delta/      | 1 year    | 1, 6           | Read often,         | Data Engineers,  |
|              | (Cleaned)   | Parquet     |           |                | append/update       | Analysts         |
|              | Gold        | Delta/      | 7 years   | 4, 5           | Read frequently,    | BI Tools,        |
|              | (Aggregated)| Parquet     |           |                | append only         | Executives       |
--------------------------------------------------------------------------------------------------------------------
| **Warehouse**| Raw Zone    | Native      | 30 days   | 2, 6           | Exploratory queries | Data Scientists  |
| (Snowflake/  |             | Tables      |           |                |                     |                  |
| BigQuery/    | Staging     | Native      | 7 days    | 1, 3, 5        | Temporary landing   | ETL Jobs         |
| Redshift)    | Zone        | Tables      |           |                | for transforms      |                  |
|              | Curated     | Native      | 2 years   | 1, 2, 4, 6     | Heavy BI querying   | All Analysts     |
|              | Zone        | Tables      |           |                |                     |                  |
|              | Aggregated  | Native      | 7 years   | 4, 5           | Fast dashboard      | Executives, Ops  |
|              | Zone        | Tables      |           |                | queries             |                  |
--------------------------------------------------------------------------------------------------------------------

### Security & Compliance Matrix

--------------------------------------------------------------------------------------------------------------------
| Scenario | Data Sensitivity | Encryption | Masking Approach   | Access Restrictions        | Audit Required |
--------------------------------------------------------------------------------------------------------------------
| 1 - PII  | High - PII       | AES-256    | Hash emails, mask  | Marketing team only,       | Full lineage   |
| CRM      |                  | at rest    | phone #s           | no raw access              |                |
--------------------------------------------------------------------------------------------------------------------
| 2 - Click-| Low - Anonymous | AES-256    | None - raw stored  | Product team, DS,          | Usage only     |
| stream   |                  | at rest    |                    | no restrictions            |                |
--------------------------------------------------------------------------------------------------------------------
| 3 - Fin- | High - Financial | AES-256    | Fuzzy matches only | Finance team only,         | Full transaction|
| ancial   |                  | at rest    | no raw amounts     | 4-eyes approval for access | log             |
--------------------------------------------------------------------------------------------------------------------
| 4 - IoT  | Low - Device IDs | AES-256    | None - device IDs  | Ops team, DS               | Minimal        |
|          |                  | at rest    | only               |                            |                |
--------------------------------------------------------------------------------------------------------------------
| 5 - HR   | Critical - Salary| AES-256 +  | Agg only - no      | HR leadership only,        | Full audit with|
|          |                  | TDE        | individuals        | quarterly access review    | sign-off       |
--------------------------------------------------------------------------------------------------------------------
| 6 - Prod | Low - Public     | AES-256    | None               | All employees read-only    | Change log only|
| Catalog  |                  | at rest    |                    |                            |                |
--------------------------------------------------------------------------------------------------------------------

### Cost Allocation by Scenario

--------------------------------------------------------------------------------------------------------------------
| Scenario | Compute Cost     | Storage Cost     | Network Cost     | Total Monthly  | % of Budget |
|          | (Est. $/month)   | (Est. $/month)    | (Est. $/month)   | Estimate       |             |
--------------------------------------------------------------------------------------------------------------------
| 1 - PII  | $150 (Spark)     | $50 (small)       | $10              | $210           | 5%          |
| CRM      |                  |                   |                  |                |             |
--------------------------------------------------------------------------------------------------------------------
| 2 - Click-| $800 (BigQuery  | $400 (raw 10M/day)| $50              | $1,250         | 30%         |
| stream   | slots)           |                   |                  |                |             |
--------------------------------------------------------------------------------------------------------------------
| 3 - Fin- | $200 (Python     | $30 (CSV +        | $10              | $240           | 6%          |
| ancial   | compute)         | PostgreSQL)       |                  |                |             |
--------------------------------------------------------------------------------------------------------------------
| 4 - IoT  | $1,200 (Streaming| $800 (50M/day raw)| $100             | $2,100         | 50%         |
|          | + Batch)         | + aggregates)     |                  |                |             |
--------------------------------------------------------------------------------------------------------------------
| 5 - HR   | $50 (monthly)    | $20 (tiny)        | $5               | $75            | 2%          |
--------------------------------------------------------------------------------------------------------------------
| 6 - Prod | $200 (CDC +      | $100 (small)      | $25              | $325           | 7%          |
| Catalog  | Snowflake)       |                   |                  |                |             |
--------------------------------------------------------------------------------------------------------------------
| **TOTAL**| **$2,600**       | **$1,400**        | **$200**         | **$4,200**     | **100%**    |
--------------------------------------------------------------------------------------------------------------------

### Monitoring & Alerting Matrix

--------------------------------------------------------------------------------------------------------------------
| Scenario | Metric Monitored      | Threshold          | Alert Channel      | Response Team     |
--------------------------------------------------------------------------------------------------------------------
| 1 - PII  | Pipeline success rate | < 98%              | Email + Teams      | Data Engineering  |
| CRM      | PII masking validation| Any failure        | PagerDuty (urgent) | Security Team     |
--------------------------------------------------------------------------------------------------------------------
| 2 - Click-| Data latency          | > 2 min            | Dashboard alert    | Streaming Ops     |
| stream   | Event count drop      | > 10% from avg     | Email              | Data Engineering  |
--------------------------------------------------------------------------------------------------------------------
| 3 - Fin- | File arrival time     | > 9 AM             | Email to finance   | Finance Ops       |
| ancial   | Unmatched rate        | > 1%               | Email + Teams      | Data Engineering  |
--------------------------------------------------------------------------------------------------------------------
| 4 - IoT  | Streaming lag         | > 1 min            | PagerDuty          | Streaming Ops     |
|          | Device offline count  | > 5% of devices    | Dashboard + Email  | IoT Team          |
--------------------------------------------------------------------------------------------------------------------
| 5 - HR   | Pipeline execution    | Job fails          | Email              | Data Engineering  |
|          | Dept count anomaly    | < 5 depts total    | Email to HR        | HR Coordinator    |
--------------------------------------------------------------------------------------------------------------------
| 6 - Prod | CDC latency           | > 10 min           | Email              | Data Engineering  |
| Catalog  | Merge failures        | > 0                | Teams notification | Integration Team  |
--------------------------------------------------------------------------------------------------------------------

### Disaster Recovery & Business Continuity

--------------------------------------------------------------------------------------------------------------------
| Scenario | RPO (Recovery Point) | RTO (Recovery Time) | Backup Strategy     | Failover Method   |
--------------------------------------------------------------------------------------------------------------------
| 1 - PII  | 24 hours             | 4 hours             | Daily snapshots     | Active-Passive    |
| CRM      |                      |                     | + transaction logs  |                   |
--------------------------------------------------------------------------------------------------------------------
| 2 - Click-| 1 hour               | 1 hour              | Kafka offset +      | Active-Active     |
| stream   |                      |                     | BigQuery time travel| Multi-region      |
--------------------------------------------------------------------------------------------------------------------
| 3 - Fin- | 24 hours             | 8 hours             | Full daily backups  | Point-in-time     |
| ancial   |                      |                     | + WAL archiving     | restore           |
--------------------------------------------------------------------------------------------------------------------
| 4 - IoT  | 15 minutes           | 30 minutes          | Dual writes to      | Active-Active     |
|          |                      |                     | secondary region    | Multi-region      |
--------------------------------------------------------------------------------------------------------------------
| 5 - HR   | 1 month              | 24 hours            | Monthly exports     | Restore from      |
|          |                      |                     | + quarterly archive | backup            |
--------------------------------------------------------------------------------------------------------------------
| 6 - Prod | 5 minutes            | 15 minutes          | CDC replay from     | Standby cluster   |
| Catalog  |                      |                     | source DB           |                   |
--------------------------------------------------------------------------------------------------------------------

### Summary: Architecture Decision Matrix

--------------------------------------------------------------------------------------------------------------------
| Decision Point              | ETL Path (Scenarios 1,3,5) | ELT Path (Scenarios 2,6) | Hybrid Path (Scenario 4) |
--------------------------------------------------------------------------------------------------------------------
| **Primary Driver**          | Compliance / Complex Logic | Exploration / Simplicity  | Multiple Consumer Needs  |
| **Transform Location**      | External (Python/Spark)    | In-warehouse (SQL)        | Both                     |
| **Raw Data in Warehouse?**  | NO - masked/agg only       | YES - full raw            | Partial - aggregates only|
| **Compute Engine**          | Databricks / Python        | BigQuery / Snowflake      | Spark + Snowflake        |
| **Storage Primary**         | Data Lake → Warehouse      | Warehouse (raw + curated) | Data Lake + Warehouse    |
| **Latency Tolerance**       | Hours (batch)              | Minutes (streaming)       | Seconds (real-time)      |
| **Main Complexity**         | Data privacy / algorithms | Query optimization        | Pipeline synchronization |
| **Key Success Metric**      | Zero compliance breaches  | Query performance + freshness| Dual delivery accuracy  |
--------------------------------------------------------------------------------------------------------------------

## This comprehensive hybrid architecture design provides **separate optimized paths for each scenario** 
 ##   while maintaining **unified governance, monitoring, and security controls** across the entire data platform. 
## The architecture scales from 5K records/month (HR) to 50M events/day (IoT) with appropriate technologies for each use case.